# 📊 StockGro Capstone Project
## Data-Driven Stock Analysis using Time Series Models

---

**Notebook:** `Phase 1 — Project Planning & Environment Setup`  
**Author:** [Your Name]  
**Date:** [Date]  
**Institution:** [University Name]  

---

### 📌 Notebook Objectives
1. Verify environment and package installation
2. Define project configuration
3. Validate Yahoo Finance data access
4. Preview stock data for all 10 NSE tickers
5. Create `__init__.py` files for all modules
6. Run quick sanity checks before Phase 2

## 1. Environment Verification

In [1]:
# ============================================================
# CELL 1: Package Import & Version Verification
# ============================================================

import sys
import importlib

# ─── Required packages and their import names ───────────────
REQUIRED_PACKAGES = {
    'pandas':       'pandas',
    'numpy':        'numpy',
    'matplotlib':   'matplotlib',
    'seaborn':      'seaborn',
    'plotly':       'plotly',
    'yfinance':     'yfinance',
    'statsmodels':  'statsmodels',
    'pmdarima':     'pmdarima',
    'prophet':      'prophet',
    'arch':         'arch',
    'sklearn':      'sklearn',
    'tensorflow':   'tensorflow',
    'scipy':        'scipy',
    'yaml':         'yaml',
}

print(f"Python: {sys.version}")
print("=" * 50)

all_ok = True
for pkg_name, import_name in REQUIRED_PACKAGES.items():
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, '__version__', 'N/A')
        print(f"  ✅ {pkg_name:<20} v{version}")
    except ImportError:
        print(f"  ❌ {pkg_name:<20} NOT INSTALLED")
        all_ok = False

print("=" * 50)
if all_ok:
    print("🎉 All packages installed and ready!")
else:
    print("⚠️  Some packages missing. Run: pip install -r requirements.txt")

Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
  ✅ pandas               v3.0.2
  ✅ numpy                v2.4.4
  ✅ matplotlib           v3.10.9
  ✅ seaborn              v0.13.2
  ✅ plotly               v6.7.0
  ✅ yfinance             v1.3.0
  ✅ statsmodels          v0.14.6
  ✅ pmdarima             v2.1.1


d:\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✅ prophet              v1.3.0
  ✅ arch                 v8.0.0
  ✅ sklearn              v1.8.0
  ✅ tensorflow           v2.21.0
  ✅ scipy                v1.17.1
  ✅ yaml                 v6.0.2
🎉 All packages installed and ready!


## 2. Project Configuration

In [2]:
# ============================================================
# CELL 2: Project Configuration — Core Settings
# ============================================================

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ─── Set working directory to project root ──────────────────
# Uncomment and update path if running from a different location
# os.chdir('/path/to/StockGro_Capstone')

# ─── Date Configuration ─────────────────────────────────────
CONFIG = {
    # Data range
    'FULL_START':  '2021-01-01',
    'FULL_END':    '2025-12-31',
    
    # Train/Test split
    'TRAIN_START': '2021-01-01',
    'TRAIN_END':   '2025-06-30',
    'TEST_START':  '2025-07-01',
    'TEST_END':    '2025-12-31',
    
    # Forecast
    'FORECAST_DAYS': 5,
    
    # Portfolio
    'TOTAL_CAPITAL_INR': 1_000_000,  # ₹10,00,000
    'RISK_FREE_RATE':    0.06,        # 6% annual (10-yr GOI)
    
    # Model
    'LSTM_LOOKBACK': 60,
    'RANDOM_SEED':   42,
}

# ─── Stock Universe ──────────────────────────────────────────
STOCKS = {
    'HDFCBANK.NS':  {'name': 'HDFC Bank',              'sector': 'Banking',  'role': 'Core'},
    'ICICIBANK.NS': {'name': 'ICICI Bank',              'sector': 'Banking',  'role': 'Core'},
    'INFY.NS':      {'name': 'Infosys',                 'sector': 'IT',       'role': 'Core'},
    'TCS.NS':       {'name': 'Tata Consultancy',        'sector': 'IT',       'role': 'Core'},
    'SUNPHARMA.NS': {'name': 'Sun Pharmaceutical',      'sector': 'Pharma',   'role': 'Satellite'},
    'DRREDDY.NS':   {'name': "Dr. Reddy's Labs",        'sector': 'Pharma',   'role': 'Satellite'},
    'HINDUNILVR.NS':{'name': 'Hindustan Unilever',      'sector': 'FMCG',     'role': 'Satellite'},
    'ITC.NS':       {'name': 'ITC Ltd',                 'sector': 'FMCG',     'role': 'Satellite'},
    'MARUTI.NS':    {'name': 'Maruti Suzuki',           'sector': 'Auto',     'role': 'Tactical'},
    'TATAMOTORS.NS':{'name': 'Tata Motors',             'sector': 'Auto',     'role': 'Tactical'},
}

TICKERS = list(STOCKS.keys())

# Display configuration
print("📋 PROJECT CONFIGURATION")
print("=" * 50)
print(f"  Full Period:   {CONFIG['FULL_START']} → {CONFIG['FULL_END']}")
print(f"  Training:      {CONFIG['TRAIN_START']} → {CONFIG['TRAIN_END']}")
print(f"  Testing:       {CONFIG['TEST_START']} → {CONFIG['TEST_END']}")
print(f"  Forecast Days: {CONFIG['FORECAST_DAYS']}")
print(f"  Capital:       ₹{CONFIG['TOTAL_CAPITAL_INR']:,}")
print(f"  Risk-free:     {CONFIG['RISK_FREE_RATE']*100:.0f}%")
print("=" * 50)
print(f"\n📈 STOCK UNIVERSE ({len(TICKERS)} stocks across 5 sectors)")

stocks_df = pd.DataFrame(STOCKS).T.reset_index()
stocks_df.columns = ['Ticker', 'Company', 'Sector', 'Role']
display(stocks_df)

📋 PROJECT CONFIGURATION
  Full Period:   2021-01-01 → 2025-12-31
  Training:      2021-01-01 → 2025-06-30
  Testing:       2025-07-01 → 2025-12-31
  Forecast Days: 5
  Capital:       ₹1,000,000
  Risk-free:     6%

📈 STOCK UNIVERSE (10 stocks across 5 sectors)


,Ticker,Company,Sector,Role
0,HDFCBANK.NS,HDFC Bank,Banking,Core
1,ICICIBANK.NS,ICICI Bank,Banking,Core
2,INFY.NS,Infosys,IT,Core
3,TCS.NS,Tata Consultancy,IT,Core
4,SUNPHARMA.NS,Sun Pharmaceutical,Pharma,Satellite
5,DRREDDY.NS,Dr. Reddy's Labs,Pharma,Satellite
6,HINDUNILVR.NS,Hindustan Unilever,FMCG,Satellite
7,ITC.NS,ITC Ltd,FMCG,Satellite
8,MARUTI.NS,Maruti Suzuki,Auto,Tactical
9,TATAMOTORS.NS,Tata Motors,Auto,Tactical


## 3. Yahoo Finance Connectivity Test

In [3]:
import yfinance as yf
from datetime import datetime

print("🌐 Testing Yahoo Finance connectivity...\n")

test_ticker = 'TCS.NS'
test_data = yf.download(
    tickers=test_ticker,
    start='2025-01-01',
    end='2025-01-31',
    progress=False,
    auto_adjust=False,
    group_by='column'
)

if not test_data.empty:
    print("  ✅ Connection successful!")
    print(f"  📊 {test_ticker}: {len(test_data)} trading days fetched")
    print(f"  📅 Date range: {test_data.index[0].date()} → {test_data.index[-1].date()}")

    latest_close = test_data['Close'].iloc[-1]
    if hasattr(latest_close, 'iloc'):
        latest_close = latest_close.iloc[0]

    print(f"  💰 Latest close: ₹{latest_close:.2f}")
    print(f"  📋 Columns: {list(test_data.columns)}")
    print()

    display(test_data.tail(3))
else:
    print("  ❌ Connection failed or no data returned")
    print("  → Check internet connection")
    print("  → Try: pip install --upgrade yfinance")

🌐 Testing Yahoo Finance connectivity...



  ✅ Connection successful!
  📊 TCS.NS: 22 trading days fetched
  📅 Date range: 2025-01-01 → 2025-01-30
  💰 Latest close: ₹4100.05
  📋 Columns: [('Adj Close', 'TCS.NS'), ('Close', 'TCS.NS'), ('High', 'TCS.NS'), ('Low', 'TCS.NS'), ('Open', 'TCS.NS'), ('Volume', 'TCS.NS')]



Price,Adj Close,Close,High,Low,Open,Volume
Ticker,TCS.NS,TCS.NS,TCS.NS,TCS.NS,TCS.NS,TCS.NS
Date,,,,,,
2025-01-28,3905.335205,4040.300049,4102.000000,4028.300049,4070.800049,2468272
2025-01-29,3962.847168,4099.799805,4107.850098,4054.000000,4054.000000,1217089
2025-01-30,3963.088867,4100.049805,4129.899902,4058.500000,4099.899902,1531251


In [4]:
import yfinance as yf

data = yf.download('TATAMTRDVR.NS', period='1mo', progress=False)
print(data.tail())

$TATAMTRDVR.NS: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['TATAMTRDVR.NS']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")


Empty DataFrame
Columns: [(Adj Close, TATAMTRDVR.NS), (Close, TATAMTRDVR.NS), (High, TATAMTRDVR.NS), (Low, TATAMTRDVR.NS), (Open, TATAMTRDVR.NS), (Volume, TATAMTRDVR.NS)]
Index: []


## 4. Quick Data Preview — All 10 Stocks

In [5]:
# ============================================================
# CELL 4: Quick Preview — Latest Price for All Stocks
# ============================================================

import yfinance as yf
import pandas as pd
import warnings
import time
import logging
from contextlib import redirect_stdout, redirect_stderr
import io

warnings.filterwarnings('ignore')
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

print("📊 Fetching latest data for all stocks (Dec 2024–Jan 2025)...")
print("(Full download happens in Phase 2 notebook)\n")

preview_results = []
failed_tickers = []

for ticker, info in STOCKS.items():
    data = pd.DataFrame()

    # Retry up to 3 times because Yahoo Finance can intermittently fail
    for attempt in range(3):
        try:
            # Suppress noisy Yahoo Finance messages
            with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
                data = yf.download(
                    tickers=ticker,
                    start='2024-12-01',
                    end='2025-01-31',
                    progress=False,
                    timeout=10,
                    auto_adjust=False,
                    threads=False
                )

            if not data.empty:
                break

        except Exception:
            pass

        time.sleep(1)  # brief pause before retry

    # --------------------------------------------------------
    # Process successful downloads
    # --------------------------------------------------------
    if not data.empty and len(data) > 0:
        try:
            # Extract latest close price
            latest_close = data['Close'].iloc[-1]
            if hasattr(latest_close, 'iloc'):
                latest_close = latest_close.iloc[0]
            latest_close = float(latest_close)

            # Compute daily returns
            daily_ret = data['Close'].pct_change().dropna()

            # Handle MultiIndex output from yfinance
            if isinstance(daily_ret, pd.DataFrame):
                daily_ret = daily_ret.iloc[:, 0]

            # Annualized volatility
            vol_30d = daily_ret.std() * (252 ** 0.5) * 100

            preview_results.append({
                'Ticker':         ticker,
                'Company':        info['name'],
                'Sector':         info['sector'],
                'Role':           info['role'],
                'Latest_Close':   f"₹{latest_close:,.2f}",
                'Days_Available': len(data),
                'Ann_Vol_%':      f"{float(vol_30d):.1f}%",
                'Status':         '✅ OK',
            })

        except Exception:
            preview_results.append({
                'Ticker':  ticker,
                'Company': info['name'],
                'Sector':  info['sector'],
                'Status':  '❌ PROCESSING ERROR',
            })
            failed_tickers.append(ticker)

    # --------------------------------------------------------
    # Handle failed downloads
    # --------------------------------------------------------
    else:
        preview_results.append({
            'Ticker':  ticker,
            'Company': info['name'],
            'Sector':  info['sector'],
            'Status':  '❌ NO DATA',
        })
        failed_tickers.append(ticker)

# ============================================================
# Display Results
# ============================================================

preview_df = pd.DataFrame(preview_results)
display(preview_df)

ok_count = len([r for r in preview_results if '✅' in r.get('Status', '')])

print(f"\n✅ {ok_count}/{len(STOCKS)} stocks accessible from Yahoo Finance")

if failed_tickers:
    print(f"\n⚠️ Failed tickers: {', '.join(failed_tickers)}")
    print("\n📝 TROUBLESHOOTING:")
    print("  • Yahoo Finance occasionally fails temporarily")
    print("  • Retry after a few minutes")
    print("  • Verify ticker symbols")
    print("  • Update yfinance: pip install --upgrade yfinance")

📊 Fetching latest data for all stocks (Dec 2024–Jan 2025)...
(Full download happens in Phase 2 notebook)



,Ticker,Company,Sector,Role,Latest_Close,Days_Available,Ann_Vol_%,Status
0,HDFCBANK.NS,HDFC Bank,Banking,Core,₹845.75,43.0,17.5%,✅ OK
1,ICICIBANK.NS,ICICI Bank,Banking,Core,"₹1,255.60",43.0,16.8%,✅ OK
2,INFY.NS,Infosys,IT,Core,"₹1,859.95",43.0,25.9%,✅ OK
3,TCS.NS,Tata Consultancy,IT,Core,"₹4,100.05",43.0,24.6%,✅ OK
4,SUNPHARMA.NS,Sun Pharmaceutical,Pharma,Satellite,"₹1,739.10",43.0,19.9%,✅ OK
5,DRREDDY.NS,Dr. Reddy's Labs,Pharma,Satellite,"₹1,194.50",43.0,24.5%,✅ OK
6,HINDUNILVR.NS,Hindustan Unilever,FMCG,Satellite,"₹2,408.75",43.0,18.7%,✅ OK
7,ITC.NS,ITC Ltd,FMCG,Satellite,₹436.20,43.0,18.5%,✅ OK
8,MARUTI.NS,Maruti Suzuki,Auto,Tactical,"₹12,000.00",43.0,22.0%,✅ OK
9,TATAMOTORS.NS,Tata Motors,Auto,NaN,NaN,NaN,NaN,❌ NO DATA



✅ 9/10 stocks accessible from Yahoo Finance

⚠️ Failed tickers: TATAMOTORS.NS

📝 TROUBLESHOOTING:
  • Yahoo Finance occasionally fails temporarily
  • Retry after a few minutes
  • Verify ticker symbols
  • Update yfinance: pip install --upgrade yfinance


## 5. Folder Structure Verification

In [6]:
# ============================================================
# CELL 5: Verify Project Folder Structure
# ============================================================

import os

REQUIRED_DIRS = [
    'data/raw', 'data/processed', 'data/features',
    'notebooks',
    'models/arima', 'models/ets', 'models/prophet',
    'models/lstm', 'models/garch', 'models/ensemble',
    'outputs/charts', 'outputs/reports',
    'outputs/predictions', 'outputs/portfolio',
    'src/data', 'src/models', 'src/evaluation',
    'src/portfolio', 'src/utils', 'src/visualization',
    'dashboard', 'docs', 'configs', 'logs',
]

print("📁 Project Folder Structure Check")
print("=" * 40)

all_present = True
for d in REQUIRED_DIRS:
    if os.path.isdir(d):
        print(f"  ✅  {d}")
    else:
        print(f"  ❌  {d}  ← MISSING")
        all_present = False
        os.makedirs(d, exist_ok=True)
        print(f"      → Created automatically")

print("=" * 40)
if all_present:
    print("🎉 All directories present!")
else:
    print("⚠️  Missing directories were created.")

📁 Project Folder Structure Check
  ❌  data/raw  ← MISSING
      → Created automatically
  ❌  data/processed  ← MISSING
      → Created automatically
  ❌  data/features  ← MISSING
      → Created automatically
  ❌  notebooks  ← MISSING
      → Created automatically
  ❌  models/arima  ← MISSING
      → Created automatically
  ❌  models/ets  ← MISSING
      → Created automatically
  ❌  models/prophet  ← MISSING
      → Created automatically
  ❌  models/lstm  ← MISSING
      → Created automatically
  ❌  models/garch  ← MISSING
      → Created automatically
  ❌  models/ensemble  ← MISSING
      → Created automatically
  ❌  outputs/charts  ← MISSING
      → Created automatically
  ❌  outputs/reports  ← MISSING
      → Created automatically
  ❌  outputs/predictions  ← MISSING
      → Created automatically
  ❌  outputs/portfolio  ← MISSING
      → Created automatically
  ❌  src/data  ← MISSING
      → Created automatically
  ❌  src/models  ← MISSING
      → Created automatically
  ❌  src/evalu

## 6. Phase 1 Summary

In [7]:
# ============================================================
# CELL 6: Phase 1 Completion Summary
# ============================================================

from datetime import datetime

print("=" * 60)
print("  ✅ PHASE 1 COMPLETE — PROJECT PLANNING")
print("=" * 60)
print(f"  Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()
print("  DELIVERABLES:")
print("  ✅ Project configuration defined")
print("  ✅ 10 NSE stocks selected across 5 sectors")
print("  ✅ Date splits configured (train/test)")
print("  ✅ Yahoo Finance connectivity verified")
print("  ✅ Folder structure verified")
print("  ✅ requirements.txt created")
print("  ✅ project_config.yaml created")
print("  ✅ Submission checklist created")
print()
print("  NEXT STEP:")
print("  → Open notebooks/00_setup_and_data.ipynb")
print("  → Phase 2: Data Acquisition & EDA")
print("=" * 60)

  ✅ PHASE 1 COMPLETE — PROJECT PLANNING
  Completed at: 2026-05-18 09:40:44

  DELIVERABLES:
  ✅ Project configuration defined
  ✅ 10 NSE stocks selected across 5 sectors
  ✅ Date splits configured (train/test)
  ✅ Yahoo Finance connectivity verified
  ✅ Folder structure verified
  ✅ requirements.txt created
  ✅ project_config.yaml created
  ✅ Submission checklist created

  NEXT STEP:
  → Open notebooks/00_setup_and_data.ipynb
  → Phase 2: Data Acquisition & EDA
